### Common issues faced while working with CSV data

In [1]:
import os

def check_directory_access(path):
    abs_path = os.path.abspath(path)
    print(f"Checking path: {abs_path}")
    
    # Check if path exists
    if os.path.exists(abs_path):
        print("✓ Path exists")
    else:
        print("✗ Path does not exist")
        try:
            os.makedirs(abs_path)
            print("  → Created directory")
        except Exception as e:
            print(f"  → Failed to create directory: {e}")
    
    # Check if it's a directory
    if os.path.isdir(abs_path):
        print("✓ Path is a directory")
    else:
        print("✗ Path is not a directory")
    
    # Check read permission
    if os.access(abs_path, os.R_OK):
        print("✓ Read permission granted")
    else:
        print("✗ No read permission")
    
    # Check write permission
    if os.access(abs_path, os.W_OK):
        print("✓ Write permission granted")
    else:
        print("✗ No write permission")
    
    # Check execute permission (needed for directories)
    if os.access(abs_path, os.X_OK):
        print("✓ Execute permission granted")
    else:
        print("✗ No execute permission")

# Check both directories
print("Checking input directory:")
check_directory_access("../data/input")
print("Checking output directory:")
check_directory_access("../data/output")
print("\nChecking checkpoint directory:")
check_directory_access("../data/checkpoints")

Checking input directory:
Checking path: /opt/workspace/data/input
✓ Path exists
✓ Path is a directory
✓ Read permission granted
✓ Write permission granted
✓ Execute permission granted
Checking output directory:
Checking path: /opt/workspace/data/output
✓ Path exists
✓ Path is a directory
✓ Read permission granted
✓ Write permission granted
✓ Execute permission granted

Checking checkpoint directory:
Checking path: /opt/workspace/data/checkpoints
✓ Path exists
✓ Path is a directory
✓ Read permission granted
✓ Write permission granted
✓ Execute permission granted


In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = (SparkSession
         .builder
         .appName("read-csv-data-02")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "512m")
         .getOrCreate())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/02/24 10:03:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark.sparkContext.setLogLevel("ERROR")

In [5]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

In [6]:
 schema = StructType([
    StructField("show_id", StringType(), True),
    StructField("type", StringType(), True),
    StructField("title", StringType(), True),
    StructField("director", StringType(), True),
    StructField("cast", StringType(), True),
    StructField("country", StringType(), True),
    StructField("date_added", DateType(), True),
    StructField("release_year", IntegerType(), True),
    StructField("rating", StringType(), True),
    StructField("duration", StringType(), True),
    StructField("listed_in", StringType(), True),
    StructField("description", StringType(), True),
])

In [7]:
schema

StructType([StructField('show_id', StringType(), True), StructField('type', StringType(), True), StructField('title', StringType(), True), StructField('director', StringType(), True), StructField('cast', StringType(), True), StructField('country', StringType(), True), StructField('date_added', DateType(), True), StructField('release_year', IntegerType(), True), StructField('rating', StringType(), True), StructField('duration', StringType(), True), StructField('listed_in', StringType(), True), StructField('description', StringType(), True)])

In [8]:
csv_path="/opt/workspace/data/input/"

In [9]:
%whos

Variable                 Type                 Data/Info
-------------------------------------------------------
DateType                 DataTypeSingleton    <class 'pyspark.sql.types.DateType'>
IntegerType              DataTypeSingleton    <class 'pyspark.sql.types.IntegerType'>
SparkSession             type                 <class 'pyspark.sql.session.SparkSession'>
StringType               DataTypeSingleton    <class 'pyspark.sql.types.StringType'>
StructField              type                 <class 'pyspark.sql.types.StructField'>
StructType               type                 <class 'pyspark.sql.types.StructType'>
check_directory_access   function             <function check_directory<...>access at 0x7f6602cab490>
csv_path                 str                  /opt/workspace/data/input/
os                       module               <module 'os' from '/usr/lib/python3.10/os.py'>
schema                   StructType           StructType([StructField('<...>n', StringType(), True)])
spar

In [10]:
# df = spark \
#     .readStream \
#     .schema(schema) \
#     .option("header", "true") \
#     .csv(csv_path)

In [25]:
df = spark.readStream \
    .format("csv") \
    .option("basePath", "/opt/workspace/data/input/") \
    .option("path", "/opt/workspace/data/input/") \
    .schema(schema) \
    .load()

df.writeStream.toTable("my_table", checkpointLocation="../data/checkpoints")

25/02/24 10:08:35 ERROR MicroBatchExecution: Query [id = 182bc739-1cfc-42a0-811b-b572b894af94, runId = 09f721bb-d0ef-4384-9339-b017c4b26f88] terminated with error
java.lang.IllegalArgumentException: Wrong basePath /opt/workspace/data/input for the root path: file:/opt/workspace/data/netflix_titles.csv
	at org.apache.spark.sql.execution.datasources.PartitioningAwareFileIndex.$anonfun$basePaths$3(PartitioningAwareFileIndex.scala:274)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.execution.datasources.PartitioningAwareFileIndex.basePaths(PartitioningAwareFileIndex.scala:272)
	at org.apache.spark.sql.execution.datasources.PartitioningAwareFileIndex.inferPartitioning(PartitioningAwareFileIndex.scala:197)
	at org.apache.spark.sql.execution.datasources.InMemoryFileIndex.partitionSpec(InMemoryFileIndex.scala:75)
	at org.apache.spark.sql.execution.datasources.PartitioningAwareFileIndex.partitionSchema(PartitioningAwareFileIndex.scala:51)
	at org.apache.spark.sql.execution.

In [19]:
df.show(1, truncate=False)

AnalysisException: Queries with streaming sources must be executed with writeStream.start();
FileSource[/opt/workspace/data/input/]

In [18]:
query = df \
    .writeStream \
    .outputMode("append") \
    .format("csv") \
    .option("path", "../data/output/") \
    .option("checkpointLocation", "../data/checkpoints") \
    .start()
query.awaitTermination()

25/02/24 10:05:15 ERROR MicroBatchExecution: Query [id = 182bc739-1cfc-42a0-811b-b572b894af94, runId = 661bf592-5e90-4c4d-8461-df9eceecfa0c] terminated with error
java.lang.IllegalArgumentException: Wrong basePath /opt/workspace/data/input for the root path: file:/opt/workspace/data/netflix_titles.csv
	at org.apache.spark.sql.execution.datasources.PartitioningAwareFileIndex.$anonfun$basePaths$3(PartitioningAwareFileIndex.scala:274)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.execution.datasources.PartitioningAwareFileIndex.basePaths(PartitioningAwareFileIndex.scala:272)
	at org.apache.spark.sql.execution.datasources.PartitioningAwareFileIndex.inferPartitioning(PartitioningAwareFileIndex.scala:197)
	at org.apache.spark.sql.execution.datasources.InMemoryFileIndex.partitionSpec(InMemoryFileIndex.scala:75)
	at org.apache.spark.sql.execution.datasources.PartitioningAwareFileIndex.partitionSchema(PartitioningAwareFileIndex.scala:51)
	at org.apache.spark.sql.execution.

StreamingQueryException: [STREAM_FAILED] Query [id = 182bc739-1cfc-42a0-811b-b572b894af94, runId = 661bf592-5e90-4c4d-8461-df9eceecfa0c] terminated with exception: Wrong basePath /opt/workspace/data/input for the root path: file:/opt/workspace/data/netflix_titles.csv

In [ ]:
df.printSchema()